# ArchiCrop tutorial - Cereals

This notebook is a tutorial for generating any cereal plant architecture using ArchiCrop. 


Examples are given for sorghum, maize, wheat and rice.

## 0. Imports

In [1]:
from __future__ import annotations

import os
import sys
from datetime import date
sys.path.append('../data')
import numpy as np
from oawidgets.plantgl import *  # noqa: F403

from openalea.archicrop.archicrop import ArchiCrop
from openalea.archicrop.display import build_scene, display_scene
from openalea.archicrop.stics_io import read_sti_file, read_xml_file
from openalea.plantgl.all import Color3, Material, Scene

%gui qt

In [184]:
today_str = date.today().strftime("%Y-%m-%d")
os.makedirs(f"D:/PhD_Oriane/simulations_ArchiCrop/{today_str}", exist_ok=True)

m = Material(Color3(0,80,0))

def save(name, species):
    Viewer.grids.setXYPlane(False)
    Viewer.grids.setYZPlane(False)
    Viewer.grids.setXZPlane(False)
    if name == 'leaf' or 'stem' or 'phytomer':
        Viewer.camera.lookAt((100,-700,200), (85,2000,100))
    elif name == 'axis' or 'plant':
        Viewer.camera.lookAt((200,-800,400), (85,0,200))
    elif name == 'crop':
        Viewer.camera.lookAt((200,-800,400), (85,0,200))
    # Viewer.camera.lookAt((150,0,38), (85,0,38))
    Viewer.frameGL.saveImage(f"D:/PhD_Oriane/simulations_ArchiCrop/{today_str}/{name}_{species}.png")

def view(mesh, name, species):
    if isinstance(mesh, list):
        shape = [Shape(sp,m) for sp in mesh]
        PlantGL(Scene(shape))
        Viewer.display(Scene(shape))
    else:
        shape = Shape(mesh,m)
        Viewer.display(shape)
    save(name, species)

## 1. Set plant architectural parameters

Set topological, geometrical and developmental parameters, in a range corresponding a given species, found in literature.

In [185]:
from archi_dict import archi_sorghum #, archi_maize, archi_wheat, archi_rice

archi_species = {"sorghum": archi_sorghum}

## 2. Instantiate ArchiCrop object

In [186]:
archi = archi_sorghum
species = list(archi_species.keys())[0]

stem_diameter = archi_species[species]['diam_base']
stem_length = archi_species[species]['height'] / archi_species[species]['nb_phy'] 
leaf_length = archi_species[species]['leaf_area'] / archi_species[species]['nb_phy'] / 10
insertion_angle = archi_species[species]['insertion_angle']
scurv = archi_species[species]['scurv']
curvature = archi_species[species]['curvature']
wl = archi_species[species]['wl']

stem_diameter = 6 # archi_species[species]['diam_base']
stem_length = archi_species[species]['height'] / archi_species[species]['nb_phy']

## 4. Plot 3D organs

In [187]:
from openalea.archicrop.cereal_leaf import parametric_cereal_leaf
from openalea.archicrop.geometry import leaf_mesh, stem_mesh
from openalea.plantgl.all import Vector3, Material, Shape
from oawidgets.plantgl import *
%gui qt

def simple_leaf(leaf, leaf_length, wl, stem_diameter,  ratio=1):
    total_length = leaf_length
    lw_ratio = 1/wl

    L_shape = total_length
    Lw_shape = total_length / lw_ratio
    length = total_length * ratio
    s_base = 0
    s_top = 1.0

    return leaf_mesh(leaf, L_shape, Lw_shape, length, s_base, s_top, stem_diameter=stem_diameter)

def a_leaf(species):   
    leaf = parametric_cereal_leaf(insertion_angle=insertion_angle, scurv=scurv, curvature=curvature, klig=0.6, swmax=0.55, f1=0.64, f2=0.92)
    mesh_leaf = simple_leaf(leaf, leaf_length, wl, stem_diameter) 
    
    # shape_leaf = Shape(mesh_leaf,m)
    # PlantGL(shape_leaf)
    return mesh_leaf, "leaf"

for species in archi_species:
    mesh, name = a_leaf(species)
    view(mesh, name, species)

In [188]:
def a_stem(species):
    mesh_stem = stem_mesh(length=stem_length, visible_length=stem_length, stem_diameter=stem_diameter, classic=True)
    # shape_stem = Shape(mesh_stem,m)
    # PlantGL(shape_stem)
    return mesh_stem, "stem"

for species in archi_species:
    mesh, name = a_stem(species)
    view(mesh, name, species)

## 5. Plot 3D phytomers

In [189]:
def a_phytomer(species):
    mesh = [Translated(0,0,0,a_stem(species)[0]),Translated(0,0,stem_length,a_leaf(species)[0])]
    # shape = [Shape(sp,m) for sp in s]
    # PlantGL(Scene(shape))
    return mesh, "phytomer"

for species in archi_species:
    mesh, name = a_phytomer(species)
    view(mesh, name, species)

## 6. Plot 3D stems

In [39]:
archi_stem = archi.copy()
archi_stem['nb_tillers'] = 0
archi_stem['leaf_area'] = archi['leaf_area'] / (archi['nb_tillers']+1)
axis = ArchiCrop(**archi_stem)
axis.generate_potential_plant()
g_axis = axis.g
scene, _ = build_scene(g_axis, leaf_material = m, stem_material = m)
Viewer.display(scene)
save(name='axis', species=species)

## 7. Plot 3D plants

In [9]:
plant = ArchiCrop(**archi)
plant.generate_potential_plant()
g = plant.g
scene, _ = build_scene(g, leaf_material = m, stem_material = m)
Viewer.display(scene)
save(name='plant', species=species)

## 8. Plot 3D crops

In [10]:
from openalea.archicrop.stand import agronomic_plot

density = 5.4 / 3
inter_row = 0.7

nplants, positions, domain, domain_area, unit = agronomic_plot(length=5, width=5, density=density, inter_row=inter_row, noise=0.1)
scene, labels = build_scene(g, positions, leaf_material=m, stem_material=m, senescence=False)
Viewer.display(scene)
save(name='crop', species=species)